In [ ]:
# ==============================================================================
# CELL 1 / PHASE 1: ASSOCIATION OF RELIGION DATA ARCHIVES (ARDA) - US GROUPS
# 
# ARCHITECTURE NOTE: ARDA provides a granular taxonomy of US religious groups.
# They do not provide a bulk LOD file. This script uses Strategy B (Scraping) 
# to scan the 'AllGroups' table to harvest IDs, Traditions, and Families, and 
# visits each landing page to extract the 'articleBody' Description. 
# 
# SSSOM ALIGNMENT STRATEGY: 
# 1. Hierarchy_Path: [Family] > [Group Name] (Traditions are excluded from path).
# 2. Description: [Tradition] | [Family] | [Native Profile Text]
# 3. CURIE: Synthesized as ARDA:[gid] to serve as a stable SSSOM primary key.
#
# TEXT NORMALIZATION RULE: ARDA descriptions contain embedded HTML formatting, 
# non-breaking spaces, and multi-paragraph carriage returns. This script aggressively 
# flattens all whitespace (`\s+`) into single spaces and strips leading/trailing 
# quotes to ensure the final CSV is strictly flat and machine-readable.
# ==============================================================================

import os
import sys
import re
import requests
import pandas as pd
import time
from bs4 import BeautifulSoup
from dotenv import load_dotenv

# --- 1. Environment & Central Schema Setup ---
PILOT_LIMIT = None  # Set to None for full crawl later

notebook_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
config_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "config"))
raw_data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "raw"))
os.makedirs(raw_data_dir, exist_ok=True)

load_dotenv(os.path.join(config_dir, ".env")) 
sys.path.append(config_dir)
try:
    from ingest_schema_manager import get_registry_info, finalize_row, COLUMN_ORDER
except ImportError:
    raise ImportError("CRITICAL: Could not find ingest_schema_manager.py in the config directory.")

# --- 2. Registry Lookup & Target Setup ---
SOURCE_PREFIX = "ARDA"
registry_data = get_registry_info(
    prefix=SOURCE_PREFIX, 
    config_dir=config_dir,
    fallback_name="ARDA US Religious Groups",
    fallback_uri="https://www.thearda.com/"
)
SOURCE_NAME = registry_data['Source_Name']
raw_ingest_file = os.path.join(raw_data_dir, f"raw_{SOURCE_PREFIX.lower()}.csv")

DOMAIN = "https://www.thearda.com"
INDEX_URL = f"{DOMAIN}/us-religion/group-profiles/groups"
HEADERS = {'User-Agent': 'ReligiousMappingProject/1.0'}

# --- 3. Persistent Tracking Check ---
if os.path.exists(raw_ingest_file):
    existing_df = pd.read_csv(raw_ingest_file, dtype={'Concept_ID': str})
    if list(existing_df.columns) != COLUMN_ORDER:
        print(f"[!] Schema mismatch detected. Deleting outdated {os.path.basename(raw_ingest_file)}...")
        os.remove(raw_ingest_file)
        processed_ids = set()
    else:
        processed_ids = set(existing_df['Concept_ID'].tolist())
        print(f"[*] Resuming: {len(processed_ids)} groups already processed.")
else:
    processed_ids = set()

# --- 4. Main Extraction Loop ---
print(f"[*] Fetching Group Index from {INDEX_URL}...")
try:
    response = requests.get(INDEX_URL, headers=HEADERS, timeout=15)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'html.parser')
    
    table = soup.find('table', id='AllGroups')
    if not table:
        print("[!] Could not find table with id='AllGroups'.")
        sys.exit(1)
        
    tbody = table.find('tbody')
    rows = tbody.find_all('tr') if tbody else table.find_all('tr')[1:]

    count = 0
    for row in rows:
        if PILOT_LIMIT and count >= PILOT_LIMIT:
            break
            
        cols = row.find_all('td')
        if len(cols) < 3: continue
        
        name_link = cols[0].find('a')
        if not name_link: continue
        
        relative_href = name_link.get('href', '')
        gid_match = re.search(r'[D|gid]=(\d+)', relative_href)
        if not gid_match: continue
            
        gid = gid_match.group(1)
        if gid in processed_ids: continue

        group_name = name_link.text.strip()
        tradition = cols[1].text.strip() if cols[1].text else "Unclassified"
        family = cols[2].text.strip() if cols[2].text else "Unclassified"
        
        # Assign Landing Page URL as URI
        landing_page_url = f"{DOMAIN}{relative_href}"
        
        print(f"[{count+1}] Fetching details for: {group_name} (ID:{gid})...")
        
        # Nested Crawl for Description
        profile_text = ""
        try:
            p_res = requests.get(landing_page_url, headers=HEADERS, timeout=10)
            if p_res.status_code == 200:
                p_soup = BeautifulSoup(p_res.text, 'html.parser')
                body = p_soup.find('div', itemprop='articleBody')
                if body:
                    desc_tag = body.find('b', string=re.compile(r'Description:', re.I))
                    if desc_tag:
                        raw_text = desc_tag.next_sibling
                        if raw_text:
                            # Clean whitespace and strip stray quotes
                            clean_text = str(raw_text).strip()
                            clean_text = re.sub(r'\s+', ' ', clean_text)
                            profile_text = clean_text.strip('"').strip("'")
        except Exception as e:
            print(f"    [!] Failed to fetch profile page for {gid}: {e}")
        
        hierarchy_path = f"{family} > {group_name}"
        final_description = f"{tradition} | {family} | {profile_text if profile_text else ''}"
        
        extracted_data = {
            'Source_System': SOURCE_NAME,
            'Primary_Label': group_name,
            'CURIE': f"{SOURCE_PREFIX}:{gid}",
            'Formal_Label': "",
            'Concept_Type': 'skos:Concept',
            'Hierarchy_Path': hierarchy_path,
            'Synonyms': "",
            'Description': final_description.strip(),
            'Parent_IDs': family, 
            'Concept_ID': gid,
            'URI': landing_page_url, 
            'Status': "active",
            'Crosswalks': ""
        }
        
        # --- 5. Incremental Export to Bronze Layer ---
        clean_row = finalize_row(extracted_data)
        pd.DataFrame([clean_row])[COLUMN_ORDER].to_csv(
            raw_ingest_file, mode='a', index=False, 
            header=not os.path.isfile(raw_ingest_file), encoding='utf-8-sig'
        )
        processed_ids.add(gid)
        count += 1
        time.sleep(1.5) # Politeness delay for nested crawl

except Exception as e:
    print(f"[!] Error: {e}")

print(f"\n[COMPLETE] Script finished. Ingested {count} detailed ARDA records.")

In [ ]:
# ==============================================================================
# CELL 2 / PHASE 1: ASSOCIATION OF RELIGION DATA ARCHIVES (ARDA) - WORLD TREES
# 
# ARCHITECTURE NOTE: ARDA visualizes World Religions using the GoJS library.
# The structural taxonomy is embedded as raw JSON arrays inside the HTML <script>.
# 
# SSSOM ALIGNMENT STRATEGY: 
# This script extracts the nodes and edges JSON, builds a local mathematical graph, 
# and recursively resolves the Hierarchy_Path. IDs are prefixed with 'W' 
# (e.g., W5070) to prevent collisions with ARDA's US Religion database.
#
# TEXT NORMALIZATION RULE: GoJS descriptions contain embedded formatting and 
# raw carriage returns (\r\n). The script aggressively flattens all whitespace 
# (`\s+`) into single spaces and strips leading/trailing quotes to ensure the 
# final CSV is strictly flat and machine-readable.
# ==============================================================================

import os
import sys
import re
import json
import requests
import pandas as pd
import time
from bs4 import BeautifulSoup
from dotenv import load_dotenv

# --- 1. Environment & Central Schema Setup ---
notebook_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
config_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "config"))
raw_data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "raw"))
os.makedirs(raw_data_dir, exist_ok=True)

load_dotenv(os.path.join(config_dir, ".env")) 
sys.path.append(config_dir)
try:
    from ingest_schema_manager import get_registry_info, finalize_row, COLUMN_ORDER
except ImportError:
    raise ImportError("CRITICAL: Could not find ingest_schema_manager.py in the config directory.")

# --- 2. Registry Lookup & Target Setup ---
SOURCE_PREFIX = "ARDA"
registry_data = get_registry_info(
    prefix=SOURCE_PREFIX, 
    config_dir=config_dir,
    fallback_name="ARDA Religious Groups",
    fallback_uri="https://www.thearda.com/"
)
SOURCE_NAME = registry_data['Source_Name']
raw_ingest_file = os.path.join(raw_data_dir, f"raw_arda_world_trees.csv")

INDEX_URL = "https://www.thearda.com/world-religion/family-trees"
DOMAIN = "https://www.thearda.com"
HEADERS = {'User-Agent': 'ReligiousMappingProject/1.0'}

# --- 3. Persistent Tracking Check ---
if os.path.exists(raw_ingest_file):
    existing_df = pd.read_csv(raw_ingest_file, dtype={'Concept_ID': str})
    if list(existing_df.columns) != COLUMN_ORDER:
        print(f"[!] Schema mismatch detected. Deleting outdated {os.path.basename(raw_ingest_file)}...")
        os.remove(raw_ingest_file)
        processed_ids = set()
    else:
        processed_ids = set(existing_df['Concept_ID'].tolist())
        print(f"[*] Resuming: {len(processed_ids)} groups already processed.")
else:
    processed_ids = set()

# Helper Function: Ancestry Resolution
def get_absolute_path(node_key, nodes_dict, parents_map, depth=0):
    """Recursively builds the breadcrumb path from the root."""
    if depth > 15: # Prevent infinite loops in cyclic graphs
        return nodes_dict.get(node_key, {}).get("Name", "Unknown")
        
    if node_key not in parents_map or not parents_map[node_key]:
        return nodes_dict.get(node_key, {}).get("Name", "Unknown")
        
    # Polyhierarchy handling: default to the first parent for the visual path
    primary_parent = parents_map[node_key][0]
    parent_path = get_absolute_path(primary_parent, nodes_dict, parents_map, depth + 1)
    current_name = nodes_dict.get(node_key, {}).get("Name", "Unknown")
    
    return f"{parent_path} > {current_name}"

# --- 4. Main Extraction Loop ---
print(f"[*] Fetching World Religion Tree Index from {INDEX_URL}...")
try:
    response = requests.Session().get(INDEX_URL, headers=HEADERS, timeout=15)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'html.parser')
    
    # Find links to the individual family trees
    tree_links = soup.find_all('a', href=re.compile(r'family-trees\?W=1&F=\d+'))
    # Filter out the abbreviated versions
    tree_urls = [DOMAIN + a['href'] for a in tree_links if 'showAbbrev' not in a['href']]
    
    # Deduplicate URLs
    tree_urls = list(set(tree_urls))
    print(f"[*] Found {len(tree_urls)} World Religion Trees to process.")

    all_rows = []

    for url in tree_urls:
        print(f"\n[*] Processing Tree: {url}")
        tree_res = requests.get(url, headers=HEADERS, timeout=15)
        
        # Regex to target the JSON arrays inside the createDiagram() JavaScript function
        js_match = re.search(r'createDiagram\(.*?,\s*".*?",\s*(\[.*?\]),\s*(\[.*?\])', tree_res.text, re.DOTALL)
        
        if not js_match:
            print("    [!] Could not locate GoJS data array in HTML source. Skipping.")
            continue
            
        try:
            nodes_data = json.loads(js_match.group(1))
            edges_data = json.loads(js_match.group(2))
        except json.JSONDecodeError as e:
            print(f"    [!] JSON Parse Error: {e}")
            continue
            
        # Build relational maps
        nodes_dict = {n['Key']: n for n in nodes_data}
        parents_map = {}
        
        for edge in edges_data:
            child = edge['ChildKey']
            parent = edge['ParentKey']
            if child not in parents_map:
                parents_map[child] = []
            parents_map[child].append(parent)
            
        # Transform Nodes into standard schema
        for key, node in nodes_dict.items():
            cid = f"W{key}"
            if cid in processed_ids: continue
                
            name = str(node.get('Name', '')).strip()
            
            # --- Clean Description ---
            raw_desc = str(node.get('Desc', ''))
            clean_desc = re.sub(r'^.*?\)\s*:\s*', '', raw_desc).strip()
            clean_desc = re.sub(r'\s+', ' ', clean_desc)
            clean_desc = clean_desc.strip('"').strip("'").strip()
            
            # Extract Parent IDs
            p_keys = parents_map.get(key, [])
            parent_ids_str = " | ".join([f"W{p}" for p in p_keys])
            
            hierarchy_path = get_absolute_path(key, nodes_dict, parents_map)
            
            extracted_data = {
                'Source_System': SOURCE_NAME,
                'Primary_Label': name,
                'CURIE': f"{SOURCE_PREFIX}:{cid}",
                'Formal_Label': "",
                'Concept_Type': 'skos:Concept',
                'Hierarchy_Path': hierarchy_path,
                'Synonyms': "",
                'Description': clean_desc,
                'Parent_IDs': parent_ids_str, 
                'Concept_ID': cid,
                'URI': "", 
                'Status': "active",
                'Crosswalks': ""
            }
            
            clean_row = finalize_row(extracted_data)
            all_rows.append(clean_row)
            processed_ids.add(cid)
            
        print(f"    [+] Extracted {len(nodes_data)} concepts.")
        time.sleep(1) # Be polite between trees

    # --- 5. Export to Bronze Layer ---
    if all_rows:
        export_df = pd.DataFrame(all_rows)[COLUMN_ORDER]
        export_df.to_csv(
            raw_ingest_file, mode='a', index=False, 
            header=not os.path.isfile(raw_ingest_file), encoding='utf-8-sig'
        )
        print(f"\n[COMPLETE] Successfully wrote {len(all_rows)} World Religion concepts to {os.path.basename(raw_ingest_file)}.")
    else:
        print("\n[*] No new records to write.")

except Exception as e:
    print(f"[!] Critical Error: {e}")

In [ ]:
# ==============================================================================
# ASSOCIATION OF RELIGION DATA ARCHIVES (ARDA) - MEASUREMENTS
# 
# ARCHITECTURE NOTE: ARDA organizes sociological survey concepts into more than 
# 160 "Single-Item Measures". This script scrapes the index table to harvest the 
# Concept, Category, and ID (`scid`), then visits each landing page to extract 
# the conceptual definition.
# 
# SSSOM ALIGNMENT STRATEGY: 
# 1. Hierarchy_Path: [Category] > [Concept Name].
# 2. CURIE: Synthesized as ARDA:M[scid] to prevent collisions with US Groups 
#    and World Religion IDs.
#
# TEXT NORMALIZATION RULE: Descriptive text is pulled from raw HTML siblings
# directly beneath the <h1> tag. The script stops parsing when it hits the 
# italicized question count (<i>), flattens whitespace, and strips stray quotes.
# ==============================================================================

import os
import sys
import re
import requests
import pandas as pd
import time
from bs4 import BeautifulSoup
from dotenv import load_dotenv

# --- 1. Environment & Central Schema Setup ---
PILOT_LIMIT = None  # Set to an integer like 5 to test, or None for full crawl

notebook_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
config_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "config"))
raw_data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "raw"))
os.makedirs(raw_data_dir, exist_ok=True)

load_dotenv(os.path.join(config_dir, ".env")) 
sys.path.append(config_dir)
try:
    from ingest_schema_manager import get_registry_info, finalize_row, COLUMN_ORDER
except ImportError:
    raise ImportError("CRITICAL: Could not find ingest_schema_manager.py in the config directory.")

# --- 2. Registry Lookup & Target Setup ---
SOURCE_PREFIX = "ARDA"
registry_data = get_registry_info(
    prefix=SOURCE_PREFIX, 
    config_dir=config_dir,
    fallback_name="Association of Religion Data Archives",
    fallback_uri="https://www.thearda.com/"
)
SOURCE_NAME = registry_data['Source_Name']
raw_ingest_file = os.path.join(raw_data_dir, f"raw_{SOURCE_PREFIX.lower()}_measurements.csv")

DOMAIN = "https://www.thearda.com"
INDEX_URL = f"{DOMAIN}/data-archive/measurements"
HEADERS = {'User-Agent': 'ReligiousMappingProject/1.0'}

# --- 3. Persistent Tracking Check ---
if os.path.exists(raw_ingest_file):
    existing_df = pd.read_csv(raw_ingest_file, dtype={'Concept_ID': str})
    if list(existing_df.columns) != COLUMN_ORDER:
        print(f"[!] Schema mismatch detected. Deleting outdated {os.path.basename(raw_ingest_file)}...")
        os.remove(raw_ingest_file)
        processed_ids = set()
    else:
        processed_ids = set(existing_df['Concept_ID'].tolist())
        print(f"[*] Resuming: {len(processed_ids)} measurements already processed.")
else:
    processed_ids = set()

# --- 4. Main Extraction Loop ---
print(f"[*] Fetching Measurements Index from {INDEX_URL}...")
try:
    response = requests.get(INDEX_URL, headers=HEADERS, timeout=15)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'html.parser')
    
    table = soup.find('table', id='SingleItemMeasures')
    if not table:
        print("[!] Could not find table with id='SingleItemMeasures'.")
        sys.exit(1)
        
    tbody = table.find('tbody')
    rows = tbody.find_all('tr') if tbody else table.find_all('tr')[1:]

    count = 0
    for row in rows:
        if PILOT_LIMIT and count >= PILOT_LIMIT:
            break
            
        cols = row.find_all('td')
        if len(cols) < 2: continue
        
        name_link = cols[0].find('a')
        if not name_link: continue
        
        relative_href = name_link.get('href', '')
        scid_match = re.search(r'scid=(\d+)', relative_href)
        if not scid_match: continue
            
        raw_id = scid_match.group(1)
        # Prepend 'M' to prevent collisions with ARDA's other data types
        cid = f"M{raw_id}" 
        
        if cid in processed_ids: continue

        concept_name = name_link.text.strip()
        category = cols[1].text.strip() if cols[1].text else "Uncategorized"
        
        landing_page_url = f"{DOMAIN}{relative_href}"
        
        print(f"[{count+1}] Fetching details for: {concept_name} (ID:{cid})...")
        
        # Nested Crawl for Description
        profile_text = ""
        try:
            p_res = requests.get(landing_page_url, headers=HEADERS, timeout=10)
            if p_res.status_code == 200:
                p_soup = BeautifulSoup(p_res.text, 'html.parser')
                body = p_soup.find('div', itemprop='articleBody')
                
                if body:
                    h1_tag = body.find('h1')
                    if h1_tag:
                        desc_parts = []
                        # Iterate through elements immediately after the <h1>
                        for sibling in h1_tag.next_siblings:
                            # Stop when we hit the italics ("40 Questions match...") or a structural break
                            if getattr(sibling, 'name', None) in ['i', 'hr', 'div', 'table']:
                                break
                            
                            # Collect floating text nodes
                            if isinstance(sibling, str):
                                desc_parts.append(sibling)
                                
                        raw_text = " ".join(desc_parts).strip()
                        
                        # --- Text Normalization ---
                        clean_text = re.sub(r'\s+', ' ', raw_text)
                        profile_text = clean_text.strip('"').strip("'").strip()
                        
        except Exception as e:
            print(f"    [!] Failed to fetch profile page for {cid}: {e}")
        
        hierarchy_path = f"{category} > {concept_name}"
        
        extracted_data = {
            'Source_System': SOURCE_NAME,
            'Primary_Label': concept_name,
            'CURIE': f"{SOURCE_PREFIX}:{cid}",
            'Formal_Label': "",
            'Concept_Type': 'skos:Concept',
            'Hierarchy_Path': hierarchy_path,
            'Synonyms': "",
            'Description': profile_text,
            'Parent_IDs': category, 
            'Concept_ID': cid,
            'URI': landing_page_url, 
            'Status': "active",
            'Crosswalks': ""
        }
        
        # --- 5. Incremental Export to Bronze Layer ---
        clean_row = finalize_row(extracted_data)
        pd.DataFrame([clean_row])[COLUMN_ORDER].to_csv(
            raw_ingest_file, mode='a', index=False, 
            header=not os.path.isfile(raw_ingest_file), encoding='utf-8-sig'
        )
        processed_ids.add(cid)
        count += 1
        time.sleep(1.5) # Politeness delay for nested crawl

except Exception as e:
    print(f"[!] Error: {e}")

print(f"\n[COMPLETE] Script finished. Ingested {count} ARDA Measurement concepts.")